In [1]:
# to import things from script folder on this notebooks
import sys; sys.path.append('..')
from Bio import AlignIO, SeqIO
import pandas as pd
from pathlib import Path
import itertools as it
from collections import defaultdict


from src.conf import DATA_RAW, DATA_INTERIM, DATA_PROCESSED

In [2]:
prot_len = {"NSP1":180, "NSP2":638, "NSP3":1945, "NSP4":500,
            "NSP5":306, "NSP6":290, "NSP7":83, "NSP8":198,
            "NSP9":113, "NSP10":139, "NSP11":13, "NSP12":932,
            "NSP13":601, "NSP14":527, "NSP15":346, "NSP16":298,
            "Spike":1273, "NS3":275, "E":75, "M":222, "NS6":61,
            "NS7a":121, "NS7b":43, "NS8":121, "N":419}

In [3]:
sars2protein_folder = DATA_INTERIM/'sars2_proteins'

In [6]:
class FilterXchar:
    """
    Filter seqs with non-protein character `X` percentage >= threshold
    """
    def __init__(self, th=5.0):
        self.th = th
        
    def apply(self, records):
        xf = lambda item: not (item.seq.count('X')/len(item.seq))*100 >= self.th 
        return filter(xf,records)

class Filterlen:
    """
    Filter seqs with len < max seq len
    """
    def __init__(self, maxlen):
        self.maxlen = maxlen
        
    def apply(self, records):
        lf = lambda item: not (len(item.seq)) < self.maxlen
        return filter(lf,records)

class FilterCountry:
    """
    Filter by choosen country
    """
    def __init__(self, country):
        self.country = country
        
    def apply(self, records):
        lf = lambda item: item.description.split('|')[-1]==self.country
        return filter(lf,records)

In [11]:
def split_records(records, n_max=20_000):
    """ 
    Split fasta records into equal number of n_max seqs.
    Returns an iterator.
    **Max numbers of seq in GISAID dataset ~107.000
    """
    max_n_mafft = 20_000
    n_start = 0
    for i,l in enumerate(it.tee(records,6)):
        l_slice = it.islice(l,n_start,max_n_mafft)
        n_start+=n_max
        max_n_mafft +=n_max
        
        yield l_slice

In [12]:
def write_records(records, outfile):
    with open(outfile,'w') as f:
        for rec in records:
            f.write(f'>{rec.description}\n')
            f.write(f'{rec.seq}\n')

def write_splitted_records(records, filename, outdir):
    
    for i, splitr in enumerate(split_records(records, n_max=20_000),1):
        outfile = outdir/filename.name.replace(f'.fasta',f'_split{i}.fasta')
        write_records(splitr, outfile)

# Apply some filter pipelines

## Testing method 1

filters, country , and write

In [17]:
outdir=DATA_INTERIM/'sars2_country'
c = 'Brazil'
if not outdir.is_dir():
    outdir.mkdir()

for fprot in sars2protein_folder.glob('sars2_others_*'):
    pname = fprot.name[:-6].split('_')[-1]
    # filter pipeline
    filter_pipe = [Filterlen(maxlen=prot_len[pname]),FilterXchar(th=5.0),FilterCountry(country=c)]
    records = SeqIO.parse(fprot,"fasta")
    for step in filter_pipe:
        # apply filters
        records = step.apply(records)
    
    outfile = outdir/fprot.name.replace(f'.fasta',f'_{c}.fasta')
    write_records(records, outfile)
    

### Testing  method 2
 filters and Split records and write fastas

In [ ]:
outdir=DATA_INTERIM/'sars2_splits'
if not outdir.is_dir():
    outdir.mkdir()

for fprot in sars2protein_folder.glob('sars2_others_*'):
    pname = fprot.name[:-6].split('_')[-1]
    # filter pipeline
    filter_pipe = [Filterlen(maxlen=prot_len[pname]),FilterXchar(th=5.0)]
    records = SeqIO.parse(fprot,"fasta")
    for step in filter_pipe:
        # apply filters
        records = step.apply(records)    

    # split and write files
    write_splitted_records(records, fprot, outdir)
    